# TSDAE: het embeddingmodel aanpassen aan onze taal

Stap 3 uit het plan in `10_meetlat_embeddings.ipynb`. Draait lokaal.

**Welk model als basis?** Volgens het plan komt TSDAE na de MLM-stap: zet dan `BASE_MODEL = "../models/arctic-mlm"` en `OUTPUT_DIR = "../models/arctic-mlm-tsdae"`. Met het basismodel (`Snowflake/snowflake-arctic-embed-l-v2.0`) krijg je de vergelijkingsrun: TSDAE zonder MLM.

**Hoe TSDAE werkt:** van elke rapportage wordt 60% van de woorden weggehaald. Het model maakt van die beschadigde zin één embedding, en een decoder moet daaruit de originele zin terugbouwen. Dat lukt alleen als de embedding de betekenis goed vastlegt, dus het model leert de woorden en schrijfstijl van de rapportages. Er zijn geen labels nodig.

- **Trainingsdata:** de teksten uit `RECORDS`. `NOTES` blijft erbuiten, want daarmee meten we.
- **Resultaat:** een nieuw model in `models/arctic-tsdae` (staat in `.gitignore`).
- **Vereist:** `uv add "sentence-transformers[train]"` (installeert o.a. `accelerate`).

**Eerst een proefdraai** (`DRY_RUN = True`): met 1.000 rapportages zie je of alles werkt en hoe lang een stap duurt. Daarna `DRY_RUN = False` voor de echte training.

In [ ]:
import os
import random

import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.losses import DenoisingAutoEncoderLoss
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,
)
from snowflake.snowpark import Session

BASE_MODEL = "Snowflake/snowflake-arctic-embed-l-v2.0"
OUTPUT_DIR = "../models/arctic-tsdae"

# Proefdraai: eerst met weinig rapportages kijken of alles werkt en hoe lang een stap duurt
DRY_RUN = True
N_SENTENCES = 1_000 if DRY_RUN else 50_000

DEL_RATIO = 0.6  # TSDAE-paper: 60% van de woorden weghalen
MAX_SEQ_LENGTH = 128  # Rapportages zijn kort; arctic staat standaard op 8192 tokens, wat veel geheugen kost
BATCH_SIZE = 8
LEARNING_RATE = 3e-5  # TSDAE-paper
RANDOM_STATE = 0

load_dotenv()
session = Session.builder.configs(
    {
        "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        "user": os.getenv("SNOWFLAKE_USER"),
        "password": os.getenv("SNOWFLAKE_PASSWORD"),
        "warehouse": "COMPUTE_WH",
        "database": "FRUTA",
        "schema": "PUBLIC",
    }
).create()

In [ ]:
# Haal de rapportages op en neem een steekproef. De verbinding is daarna niet meer nodig.
records_df = session.sql(
    "SELECT DISTINCT NOTE FROM RECORDS WHERE NOTE IS NOT NULL AND NOTE != ''"
).to_pandas()
session.close()

texts = (
    records_df["NOTE"]
    .sample(n=min(N_SENTENCES, len(records_df)), random_state=RANDOM_STATE)
    .tolist()
)
print(f"{len(records_df)} unieke rapportages, {len(texts)} gebruikt voor training.")

In [ ]:
# Beschadig de rapportages: haal willekeurig DEL_RATIO van de woorden weg (minstens één woord blijft staan)
rng = random.Random(RANDOM_STATE)


def add_noise(text):
    words = text.split()
    kept = [word for word in words if rng.random() > DEL_RATIO]
    return " ".join(kept) if kept else rng.choice(words)


# Eerste kolom: beschadigde zin (invoer), tweede kolom: originele zin (wat de decoder moet terugbouwen)
train_dataset = Dataset.from_dict({"noisy": [add_noise(t) for t in texts], "text": texts})
pd.DataFrame(train_dataset[:5])

In [ ]:
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH

# De decoder heeft dezelfde architectuur en deelt de gewichten met het model.
# Het embeddingmodel heeft geen LM-head; die wordt nieuw aangemaakt (meldingen over MISSING kun je negeren).
loss = DenoisingAutoEncoderLoss(model, tie_encoder_decoder=True)

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir=f"{OUTPUT_DIR}-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.0,
    save_strategy="no",
    logging_steps=25,
    report_to="none",
    seed=RANDOM_STATE,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
trainer.train()

In [ ]:
# Bewaar alleen het model (de encoder); de decoder was alleen nodig voor de training
if DRY_RUN:
    print("Proefdraai: model niet opgeslagen. Zet DRY_RUN = False voor de echte training.")
else:
    model.save(OUTPUT_DIR)
    print(f"Model opgeslagen in {OUTPUT_DIR}.")

## Daarna: meten

Open `10_meetlat_embeddings.ipynb`, zet `MODEL_NAME = "../models/arctic-tsdae"` en `RUN_NAME = "tsdae-50k"`, en draai het notebook. De nieuwe meting komt naast de nulmeting in `EMBEDDING_EVAL`.

Let vooral op `QUERY_PRECISION` en `KNN_PURITY`: arctic is al getraind om te zoeken, en TSDAE kan dat ook verslechteren. Gaan `SILHOUETTE` en `NMI` omhoog en de andere twee niet omlaag, dan is het model beter.